In [1]:
pip install yfinance dash plotly pandas


[notice] A new release of pip is available: 23.1.2 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
import yfinance as yf
import dash
from dash import html, dcc
from dash.dependencies import Input, Output
import plotly.graph_objs as go
import pandas as pd

# Create Dash app
app = dash.Dash(__name__)
app.title = "Real-Time Stock Dashboard"

# Stock list for dropdown
stocks = {
    'Apple': 'AAPL',
    'Google': 'GOOGL',
    'Microsoft': 'MSFT',
    'Amazon': 'AMZN',
}

# Layout
app.layout = html.Div(style={'backgroundColor': '#f8f9fa', 'padding': '20px'}, children=[
    html.H1("📊 Real-Time Stock Market Dashboard", style={'textAlign': 'center'}),

    html.Div([
        html.Label("Select a Company:"),
        dcc.Dropdown(
            id='stock-dropdown',
            options=[{'label': name, 'value': symbol} for name, symbol in stocks.items()],
            value='AAPL',
            clearable=False
        ),

        html.Label("Select Time Interval:"),
        dcc.RadioItems(
            id='interval-radio',
            options=[
                {'label': '1 Day', 'value': '1d'},
                {'label': '5 Days', 'value': '5d'},
                {'label': '1 Month', 'value': '1mo'},
                {'label': '3 Months', 'value': '3mo'},
                {'label': '6 Months', 'value': '6mo'},
                {'label': '1 Year', 'value': '1y'},
            ],
            value='1mo',
            labelStyle={'display': 'inline-block', 'marginRight': '10px'}
        )
    ], style={'width': '48%', 'margin': 'auto'}),

    html.Br(),

    dcc.Graph(id='stock-graph'),

    html.Div(id='stock-info', style={'textAlign': 'center', 'marginTop': '20px'})
])

# Callback to update chart and info
@app.callback(
    [Output('stock-graph', 'figure'),
     Output('stock-info', 'children')],
    [Input('stock-dropdown', 'value'),
     Input('interval-radio', 'value')]
)
def update_stock_graph(symbol, interval):
    data = yf.Ticker(symbol)
    df = data.history(period=interval)

    # Plot
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=df.index, y=df['Close'], mode='lines', name='Close Price'))
    fig.update_layout(title=f'{symbol} Stock Price',
                      xaxis_title='Date',
                      yaxis_title='Price (USD)',
                      template='plotly_white')

    # Company info
    info = data.info
    description = html.Div([
        html.H3(info.get('shortName', symbol)),
        html.P(f"Sector: {info.get('sector', 'N/A')}"),
        html.P(f"Industry: {info.get('industry', 'N/A')}"),
        html.P(f"Market Cap: {info.get('marketCap', 'N/A'):,}"),
    ])

    return fig, description

# Run server
if __name__ == '_main_':
    app.run_server(debug=True)

In [15]:
app.run(jupyter_mode="external")

Dash app running on http://127.0.0.1:8051/
